# `src/evaluation/splits.py`

Shared stratified outer folds: every model must use exactly these splits.

In [ ]:
# Setup: work from the project root so `src` / `app` imports and relative data paths resolve,
# wherever Jupyter was started.
import os
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from __future__ import annotations

import json

import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold

from src.data.loader import load_config, resolve

In [ ]:
def outer_folds(y, config: dict | None = None) -> list[tuple[np.ndarray, np.ndarray]]:
    """Deterministic (train_idx, val_idx) list from config (seed, n folds, repeats)."""
    cfg = config or load_config()
    n, rep, seed = cfg["cv"]["outer_folds"], cfg["cv"].get("outer_repeats", 1), cfg["project"]["seed"]
    cv = (StratifiedKFold(n, shuffle=cfg["cv"]["shuffle"], random_state=seed) if rep == 1
          else RepeatedStratifiedKFold(n_splits=n, n_repeats=rep, random_state=seed))
    return [(tr, va) for tr, va in cv.split(np.zeros(len(y)), y)]

In [ ]:
def save_folds(folds, path=None) -> None:
    p = resolve(path or "results/metrics/outer_folds.json")
    p.write_text(json.dumps([{"train": tr.tolist(), "val": va.tolist()} for tr, va in folds]))